# Attendance Data Cleaning

**Project:** Student Retention & Welfare Tracker  
**Role:** Person 2 — Data Analysis & Data Science

This notebook focuses on inspecting, cleaning, standardizing, and validating
the student attendance dataset.

The attendance data contains daily school-level attendance records and may
include inconsistent dates, school IDs, and attendance anomalies.

The final cleaned dataset will be saved as:

`data/processed/attendance_clean.csv`

## Step 1 — Load and Inspect the Attendance Data

The raw attendance dataset is loaded and inspected before any transformations
are applied.

The initial inspection helps identify the dataset structure, data types,
missing values, duplicate records, inconsistent formats, and potential
attendance anomalies that need to be addressed during cleaning.

In [1]:
import pandas as pd
import numpy as np

In [2]:
attendance = pd.read_csv(
    "../data/raw/track4_student_attendance.csv"
)


In [3]:
print("Shape:", attendance.shape)

Shape: (20800, 8)


In [4]:
attendance.info()

<class 'pandas.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   record_id         20375 non-null  str  
 1   date              20800 non-null  str  
 2   school_id         20800 non-null  str  
 3   grade             20800 non-null  str  
 4   total_students    20800 non-null  int64
 5   present_students  20800 non-null  int64
 6   teacher_present   19775 non-null  str  
 7   marked_by         17316 non-null  str  
dtypes: int64(2), str(6)
memory usage: 2.0 MB


In [5]:
print("Duplicate rows:", attendance.duplicated().sum())

Duplicate rows: 800


## Step 2 — Investigate Duplicate Records

The raw attendance dataset contains duplicate rows.

Before removing duplicates, the duplicate records are inspected to determine
whether they are exact repeated records or legitimate attendance records.

Only confirmed duplicate records will be removed.

In [6]:
duplicate_attendance = attendance[
    attendance.duplicated(keep=False)
].sort_values("record_id")

print(
    "Total rows involved in duplicates:",
    len(duplicate_attendance)
)

duplicate_attendance.head(20)

Total rows involved in duplicates: 1600


,record_id,date,school_id,grade,total_students,present_students,teacher_present,marked_by
1714,ATT0000060,05.08.2025,SCH0026,1,207,158,True,Headmaster
20492,ATT0000060,05.08.2025,SCH0026,1,207,158,True,Headmaster
1340,ATT0000116,2026/01/15,SCH0578,IV,142,142,Y,Headmaster
5607,ATT0000116,2026/01/15,SCH0578,IV,142,142,Y,Headmaster
588,ATT0000120,21.10.2025,sch0279,5,160,102,True,Admin
11012,ATT0000120,21.10.2025,sch0279,5,160,102,True,Admin
953,ATT0000128,2025/07/04,S0523,I,109,65,Hai,Class Teacher
11115,ATT0000128,2025/07/04,S0523,I,109,65,Hai,Class Teacher
1500,ATT0000140,2025-04-21,SCH0044,IV,222,216,Available,Clerk
19833,ATT0000140,2025-04-21,SCH0044,IV,222,216,Available,Clerk


In [7]:
print(
    "Duplicate record IDs:",
    attendance["record_id"].duplicated().sum()
)

Duplicate record IDs: 1204


## Duplicate Record Validation

Duplicate record IDs are investigated separately from exact duplicate rows.

A repeated record ID may represent either an exact duplicate record or multiple
records containing different information. Exact duplicates can be safely
removed, while records with conflicting information require further inspection.

In [8]:
# Find record IDs that occur more than once
duplicate_ids = attendance[
    attendance["record_id"].notna()
    & attendance["record_id"].duplicated(keep=False)
]

print("Rows with duplicated record IDs:", len(duplicate_ids))
print(
    "Unique duplicated record IDs:",
    duplicate_ids["record_id"].nunique()
)

Rows with duplicated record IDs: 1560
Unique duplicated record IDs: 780


In [9]:
# Check whether duplicated record IDs have conflicting values
duplicate_id_conflicts = (
    duplicate_ids
    .groupby("record_id")
    .nunique(dropna=False)
)

duplicate_id_conflicts[
    (duplicate_id_conflicts > 1).any(axis=1)
]

,date,school_id,grade,total_students,present_students,teacher_present,marked_by
record_id,,,,,,,


In [10]:
missing_id_duplicates = attendance[
    attendance["record_id"].isna()
    & attendance.duplicated(keep=False)
]

print(
    "Duplicate rows with missing record IDs:",
    len(missing_id_duplicates)
)

missing_id_duplicates.head(20)

Duplicate rows with missing record IDs: 40


,record_id,date,school_id,grade,total_students,present_students,teacher_present,marked_by
693,NaN,30/09/2025,S0538,6,184,110,1,Admin
869,NaN,10/11/2025,SCH0311,2,251,197,Y,HM
978,NaN,10-24-2025,sch0485,III,185,170,True,NaN
1074,NaN,2026-02-21,SCH0466,6,192,141,True,Admin
1703,NaN,07/12/2025,SCH-0443,10,187,136,Haan,NaN
1772,NaN,08-31-2025,sch0231,III,222,148,True,HM
2207,NaN,09/06/2025,sch0366,II,143,128,Functional,HM
2410,NaN,10-Sep-2025,SCH0199,2,233,163,Available,HM
2549,NaN,12.10.2025,SCH-0273,2,228,220,True,Admin
2890,NaN,28.09.2025,sch0090,IV,271,172,True,HM


In [11]:
print(
    "Number of exact duplicate rows:",
    attendance.duplicated().sum()
)

print(
    "Rows with missing record IDs:",
    attendance["record_id"].isna().sum()
)

Number of exact duplicate rows: 800
Rows with missing record IDs: 425


## Duplicate Handling Decision

The raw attendance dataset contains 800 exact duplicate rows.

A separate check found 780 duplicated record IDs, representing 1,560 rows.
Each duplicated record ID appeared exactly twice and had identical attendance
information.

The remaining duplicate rows occurred among records with missing record IDs.
These records were also confirmed to be exact duplicate rows based on all
available attendance fields.

Therefore, all 800 exact duplicate rows are removed while retaining the first
occurrence of each record.

In [12]:
attendance_clean = attendance.drop_duplicates().copy()

print("Rows before removing duplicates:", len(attendance))
print("Rows after removing duplicates:", len(attendance_clean))
print("Duplicate rows remaining:", attendance_clean.duplicated().sum())

Rows before removing duplicates: 20800
Rows after removing duplicates: 20000
Duplicate rows remaining: 0


## Step 3 — Missing Value Inspection

After removing exact duplicate records, the cleaned attendance dataset is
checked for missing values.

Missing values are investigated before deciding whether they should be
retained, standardized, or reconstructed.

Attendance records should not be removed simply because descriptive fields
such as the record ID or marker information are missing.

In [13]:
missing_summary = (
    attendance_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

marked_by           3369
teacher_present      986
record_id            405
date                   0
grade                  0
school_id              0
present_students       0
total_students         0
dtype: int64

In [14]:
print(
    "Total missing values:",
    attendance_clean.isna().sum().sum()
)

Total missing values: 4760


### Missing Value Handling Decision

The attendance dataset contains missing values in `record_id`,
`teacher_present`, and `marked_by`.

These records are retained because the core attendance fields
(`date`, `school_id`, `grade`, `total_students`, and `present_students`)
are complete.

- Missing `record_id` values are retained because the attendance observation
  remains usable without an identifier.
- Missing `teacher_present` values are retained as unknown rather than being
  interpreted as teacher absence.
- Missing `marked_by` values are retained because the identity of the person
  recording attendance cannot be reliably inferred.

No rows are removed because of missing values.

In [15]:
invalid_attendance = attendance_clean[
    attendance_clean["present_students"] >
    attendance_clean["total_students"]
]

print(
    "Records where present_students > total_students:",
    len(invalid_attendance)
)

invalid_attendance.head(20)

Records where present_students > total_students: 806


,record_id,date,school_id,grade,total_students,present_students,teacher_present,marked_by
8,ATT0002175,17.05.2025,SCH0398,III,260,271,True,NaN
12,ATT0007459,26/05/2025,SCH-0403,8,181,188,Y,HM
55,ATT0010713,2025/08/17,SCH0235,V,299,315,True,Class Teacher
100,ATT0003799,11.01.2026,SCH0225,I,92,104,1,Clerk
132,ATT0007882,21/11/2025,SCH0136,6,137,145,haan,HM
142,ATT0015453,2025-06-30,SCH0202,9,56,69,True,Clerk
149,ATT0012139,2026-01-25,SCH-0417,V,160,176,H,Headmaster
174,ATT0006557,11/08/2025,S0538,3,142,159,Yes,Class Teacher
180,NaN,12-Jun-2025,sch_0231,3,128,139,True,Clerk
196,ATT0005478,08.02.2026,SCH0305,I,209,225,Y,Headmaster


In [16]:
print(
    "Records where present_students > total_students:",
    len(invalid_attendance)
)


Records where present_students > total_students: 806


In [17]:
invalid_attendance = attendance_clean[
    attendance_clean["present_students"] >
    attendance_clean["total_students"]
].copy()

invalid_attendance["excess_students"] = (
    invalid_attendance["present_students"] -
    invalid_attendance["total_students"]
)

print(
    "Total invalid records:",
    len(invalid_attendance)
)

print(
    "Maximum excess students:",
    invalid_attendance["excess_students"].max()
)

print(
    "Average excess students:",
    invalid_attendance["excess_students"].mean()
)

Total invalid records: 806
Maximum excess students: 20
Average excess students: 12.464019851116625


## Attendance Count Anomaly Detection

The attendance data contains records where `present_students` exceeds
`total_students`, which is logically impossible.

A total of 806 records were identified with this issue. The maximum excess
was 20 students, with an average excess of approximately 12.46 students.

These values are not automatically corrected because the true attendance
count cannot be determined from the available data.

Instead, the records are retained and flagged as attendance anomalies so they
can be excluded or separately analyzed when calculating reliable attendance
metrics.

In [18]:
attendance_clean["attendance_count_anomaly"] = (
    attendance_clean["present_students"] >
    attendance_clean["total_students"]
)

In [19]:
print(
    "Attendance count anomalies:",
    attendance_clean["attendance_count_anomaly"].sum()
)

Attendance count anomalies: 806


In [20]:
attendance_clean["attendance_rate"] = (
    attendance_clean["present_students"] /
    attendance_clean["total_students"]
) * 100

In [21]:
print(
    "Maximum attendance rate:",
    attendance_clean["attendance_rate"].max()
)

Maximum attendance rate: 144.1860465116279


## Step 5 — Detect Potential Proxy Attendance

Attendance records may contain proxy or suspicious attendance entries,
particularly records showing 100% attendance on Sundays or declared holidays.

The date field is first standardized so that the day of the week can be
identified reliably.

Records matching suspicious attendance patterns will be flagged rather than
deleted, preserving them for audit and dashboard analysis.

In [22]:
attendance_clean["date_clean"] = pd.to_datetime(
    attendance_clean["date"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

In [23]:
print(
    "Invalid dates:",
    attendance_clean["date_clean"].isna().sum()
)

Invalid dates: 0


In [24]:
attendance_clean["day_of_week"] = (
    attendance_clean["date_clean"].dt.day_name()
)

attendance_clean["is_sunday"] = (
    attendance_clean["date_clean"].dt.dayofweek == 6
)

In [25]:
print(
    attendance_clean["is_sunday"].value_counts()
)

is_sunday
False    17122
True      2878
Name: count, dtype: int64


In [26]:
sunday_full_attendance = attendance_clean[
    attendance_clean["is_sunday"] &
    (attendance_clean["present_students"] ==
     attendance_clean["total_students"])
]

print(
    "100% attendance records on Sundays:",
    len(sunday_full_attendance)
)

100% attendance records on Sundays: 979


### Proxy Attendance Flag

A potential proxy attendance record is defined as an attendance record with
100% attendance occurring on a Sunday.

These records are flagged rather than deleted. The flag identifies records
that require review and allows them to be excluded from selected attendance
metrics without altering the original attendance values.

In [27]:
attendance_clean["proxy_attendance_flag"] = (
    attendance_clean["is_sunday"] &
    (attendance_clean["present_students"] ==
     attendance_clean["total_students"])
)

In [28]:
print(
    "Potential proxy attendance records:",
    attendance_clean["proxy_attendance_flag"].sum()
)

Potential proxy attendance records: 979


In [29]:
overlap = attendance_clean[
    attendance_clean["proxy_attendance_flag"] &
    attendance_clean["attendance_count_anomaly"]
]

print(
    "Records flagged as both proxy and count anomaly:",
    len(overlap)
)

Records flagged as both proxy and count anomaly: 0


### Proxy Attendance Validation

A total of 979 records were flagged as potential proxy attendance because they
show 100% attendance on a Sunday.

None of these records overlap with the `present_students > total_students`
anomaly. Therefore, the two anomaly categories are treated as separate
quality indicators.

## Step 6 — School ID Standardization

School IDs are represented using inconsistent formats, including prefixes,
hyphens, underscores, lowercase characters, and numeric-only identifiers.

All school IDs are standardized to the canonical `SCH####` format to support
reliable joins across the project datasets.

In [30]:
def standardize_school_id(value):
    value = str(value).strip().upper()
    
    value = value.replace("-", "").replace("_", "")
    
    if value.startswith("SCH"):
        value = value[3:]
    elif value.startswith("S"):
        value = value[1:]
    
    if value.isdigit():
        return "SCH" + value.zfill(4)
    
    return np.nan

In [31]:
attendance_clean["school_id_clean"] = (
    attendance_clean["school_id"]
    .apply(standardize_school_id)
)

In [32]:
print(
    "Missing standardized school IDs:",
    attendance_clean["school_id_clean"].isna().sum()
)

print(
    "Unique standardized school IDs:",
    attendance_clean["school_id_clean"].nunique()
)

Missing standardized school IDs: 0
Unique standardized school IDs: 600


In [33]:
attendance_clean[
    ["school_id", "school_id_clean"]
].head(20)

,school_id,school_id_clean
0,SCH-0596,SCH0596
1,sch_0054,SCH0054
2,SCH-0433,SCH0433
3,SCH-0379,SCH0379
4,SCH0208,SCH0208
5,sch0505,SCH0505
6,SCH0128,SCH0128
7,SCH0393,SCH0393
8,SCH0398,SCH0398
9,SCH-0377,SCH0377


## Step 7 — Inspect Teacher Presence Values

The `teacher_present` field contains multiple representations of whether the
teacher was present, including Boolean values, abbreviations, and
Hindi-derived terms.

The unique values are inspected before standardization so that the mapping
rules are based on the actual values present in the dataset.

Missing values will be retained as missing because an unrecorded teacher
presence status should not automatically be interpreted as absence.

In [34]:
teacher_values = attendance_clean["teacher_present"].dropna().unique()

for value in sorted(teacher_values, key=str):
    print(repr(value))

'0'
'1'
'Available'
'Broken'
'False'
'Functional'
'H'
'Haan'
'Hai'
'Kharab'
'N'
'Nahi'
'Nahi hai'
'No'
'Not Available'
'True'
'Under Repair'
'Working'
'Y'
'Yes'
'haan'
'na'


## Teacher Presence Standardization

The `teacher_present` field contains multiple representations of teacher
presence, including Boolean values, numeric values, abbreviations, and
Hindi-derived terms.

The values are standardized as:
- `1` = Teacher recorded as present/available
- `0` = Teacher recorded as absent/not available
- `NaN` = Missing or unrecorded

The text value `na` is treated as missing rather than as teacher absence.

In [35]:
teacher_status_mapping = {
    "true": 1,
    "false": 0,
    "yes": 1,
    "no": 0,
    "y": 1,
    "n": 0,
    "1": 1,
    "0": 0,
    "haan": 1,
    "hai": 1,
    "h": 1,
    "nahi": 0,
    "nahi hai": 0,
    "working": 1,
    "functional": 1,
    "available": 1,
    "kharab": 0,
    "broken": 0,
    "not available": 0,
    "under repair": 0
}

In [36]:
def standardize_teacher_presence(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    if value == "na":
        return np.nan
    
    return teacher_status_mapping.get(value, np.nan)

In [37]:
attendance_clean["teacher_present_clean"] = (
    attendance_clean["teacher_present"]
    .apply(standardize_teacher_presence)
)

In [38]:
print(
    attendance_clean["teacher_present_clean"]
    .value_counts(dropna=False)
)

teacher_present_clean
1.0    16368
0.0     2463
NaN     1169
Name: count, dtype: int64


In [39]:
for value in attendance_clean["teacher_present_clean"].dropna().unique():
    print(value)

1.0
0.0


In [40]:
attendance_clean = attendance_clean.drop(
    columns=["teacher_present"]
)

attendance_clean = attendance_clean.rename(
    columns={
        "teacher_present_clean": "teacher_present"
    }
)

In [41]:
print(attendance_clean["teacher_present"].value_counts(dropna=False))

teacher_present
1.0    16368
0.0     2463
NaN     1169
Name: count, dtype: int64


## Step 8 — Investigate Missing Record IDs

The attendance dataset contains records with missing `record_id` values.

These records are retained because their core attendance information is
complete. Before deciding whether replacement identifiers can be generated,
the remaining fields are checked for uniqueness.

Replacement IDs will only be generated if a reliable and reproducible
identifier can be constructed without changing the original attendance data.

In [42]:
missing_record_ids = attendance_clean[
    attendance_clean["record_id"].isna()
].copy()

print(
    "Missing record IDs:",
    len(missing_record_ids)
)

missing_record_ids.head(20)

Missing record IDs: 405


,record_id,date,school_id,grade,total_students,present_students,marked_by,attendance_count_anomaly,attendance_rate,date_clean,day_of_week,is_sunday,proxy_attendance_flag,school_id_clean,teacher_present
43,NaN,2026-02-24,SCH0012,6,112,84,Class Teacher,False,75.000000,2026-02-24,Tuesday,False,False,SCH0012,1.0
175,NaN,23-Mar-2026,SCH0565,9,138,115,Clerk,False,83.333333,2026-03-23,Monday,False,False,SCH0565,1.0
180,NaN,12-Jun-2025,sch_0231,3,128,139,Clerk,True,108.593750,2025-06-12,Thursday,False,False,SCH0231,1.0
185,NaN,04-Aug-2025,sch_0406,I,273,214,HM,False,78.388278,2025-08-04,Monday,False,False,SCH0406,1.0
188,NaN,09-Jul-2025,S0322,8,202,202,Clerk,False,100.000000,2025-07-09,Wednesday,False,False,SCH0322,1.0
249,NaN,20-Oct-2025,SCH0332,2,195,189,Admin,False,96.923077,2025-10-20,Monday,False,False,SCH0332,NaN
270,NaN,26/04/2025,SCH0219,IV,257,214,Clerk,False,83.268482,2025-04-26,Saturday,False,False,SCH0219,1.0
338,NaN,17-Jun-2025,SCH0084,2,66,54,NaN,False,81.818182,2025-06-17,Tuesday,False,False,SCH0084,1.0
356,NaN,23/07/2025,SCH-0530,4,73,69,Class Teacher,False,94.520548,2025-07-23,Wednesday,False,False,SCH0530,1.0
362,NaN,20-Aug-2025,sch_0586,V,55,45,HM,False,81.818182,2025-08-20,Wednesday,False,False,SCH0586,NaN


In [43]:
duplicate_attendance_keys = (
    missing_record_ids
    .duplicated(
        subset=["date", "school_id", "grade"],
        keep=False
    )
)

print(
    "Missing-ID records sharing date + school + grade:",
    duplicate_attendance_keys.sum()
)

Missing-ID records sharing date + school + grade: 0


### Missing Record ID Decision

A total of 405 attendance records have missing `record_id` values.

The combination of `date`, `school_id`, and `grade` was checked for duplicate
records among these rows. No duplicate combinations were found.

Although these fields could be combined to create a surrogate identifier, new
values are not generated because the original record identifier cannot be
reliably reconstructed.

The records are therefore retained with missing `record_id` values because
their attendance information remains valid and usable.

## Step 9 — Inspect Attendance Marker Values

The `marked_by` field identifies the person or role responsible for recording
attendance.

The field may contain inconsistent naming conventions or abbreviations.
Unique values are inspected before standardization.

In [44]:
marked_by_values = attendance_clean["marked_by"].dropna().unique()

for value in sorted(marked_by_values, key=str):
    print(repr(value))

'Admin'
'Class Teacher'
'Clerk'
'HM'
'Headmaster'


### Marker Standardization Decision

The `marked_by` field contains five representations of attendance-recording
roles.

The abbreviation `HM` is standardized to `Headmaster` because both `HM` and
`Headmaster` represent the same role.

The other role values (`Admin`, `Class Teacher`, and `Clerk`) are retained.
Missing values remain missing.

In [45]:
marked_by_mapping = {
    "HM": "Headmaster"
}

attendance_clean["marked_by"] = (
    attendance_clean["marked_by"]
    .replace(marked_by_mapping)
)

In [46]:
print(
    attendance_clean["marked_by"].value_counts(dropna=False)
)

marked_by
Headmaster       6727
NaN              3369
Admin            3341
Clerk            3341
Class Teacher    3222
Name: count, dtype: int64


## Step 10 — Inspect Grade Values

The `grade` field may contain both numeric and Roman numeral representations
of the same school grade.

The unique values are inspected before standardization so that equivalent
representations can be mapped to a common numeric grade.

In [47]:
grade_values = attendance_clean["grade"].unique()

for value in sorted(grade_values, key=str):
    print(repr(value))

'1'
'10'
'2'
'3'
'4'
'5'
'6'
'7'
'8'
'9'
'I'
'II'
'III'
'IV'
'V'


In [48]:
print(
    attendance_clean["grade"].value_counts()
)

grade
2      1436
4      1402
V      1388
7      1380
6      1359
I      1357
IV     1356
8      1352
1      1317
3      1313
9      1306
III    1283
5      1270
II     1244
10     1237
Name: count, dtype: int64


### Grade Standardization Decision

The attendance dataset contains both numeric and Roman numeral
representations of school grades.

Roman numeral values are converted to their equivalent numeric grades:
`I → 1`, `II → 2`, `III → 3`, `IV → 4`, and `V → 5`.

Numeric grades from 1 to 10 are retained as they are.

The final grade field uses numeric values consistently.

In [50]:
grade_mapping = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5
}

attendance_clean["grade"] = (
    attendance_clean["grade"]
    .replace(grade_mapping)
)

In [51]:
attendance_clean["grade"] = pd.to_numeric(
    attendance_clean["grade"],
    errors="coerce"
)

In [52]:
print(
    attendance_clean["grade"].value_counts().sort_index()
)

grade
1     2674
2     2680
3     2596
4     2758
5     2658
6     1359
7     1380
8     1352
9     1306
10    1237
Name: count, dtype: int64


In [53]:
print(
    "Missing grades:",
    attendance_clean["grade"].isna().sum()
)

print(
    "Unique grades:",
    sorted(attendance_clean["grade"].unique())
)

Missing grades: 0
Unique grades: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [54]:
print("Total records:", len(attendance_clean))

print(
    "Attendance count anomalies:",
    attendance_clean["attendance_count_anomaly"].sum()
)

print(
    "Potential proxy attendance:",
    attendance_clean["proxy_attendance_flag"].sum()
)

print(
    "Attendance rate below 0:",
    (attendance_clean["attendance_rate"] < 0).sum()
)

print(
    "Attendance rate above 100:",
    (attendance_clean["attendance_rate"] > 100).sum()
)

print(
    "Missing attendance rates:",
    attendance_clean["attendance_rate"].isna().sum()
)

Total records: 20000
Attendance count anomalies: 806
Potential proxy attendance: 979
Attendance rate below 0: 0
Attendance rate above 100: 806
Missing attendance rates: 0


In [55]:
rate_above_100 = attendance_clean[
    attendance_clean["attendance_rate"] > 100
]

print(
    "Records with attendance rate > 100%:",
    len(rate_above_100)
)

print(
    "Of these, count anomalies:",
    rate_above_100["attendance_count_anomaly"].sum()
)

Records with attendance rate > 100%: 806
Of these, count anomalies: 806


In [58]:
attendance_final = attendance_clean.copy()

attendance_final = attendance_final.drop(
    columns=["date", "school_id"]
)

attendance_final = attendance_final.rename(
    columns={
        "date_clean": "date",
        "school_id_clean": "school_id"
    }
)

In [59]:
attendance_final.info()

<class 'pandas.DataFrame'>
Index: 20000 entries, 0 to 20799
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   record_id                 19595 non-null  str           
 1   grade                     20000 non-null  int64         
 2   total_students            20000 non-null  int64         
 3   present_students          20000 non-null  int64         
 4   marked_by                 16631 non-null  str           
 5   attendance_count_anomaly  20000 non-null  bool          
 6   attendance_rate           20000 non-null  float64       
 7   date                      20000 non-null  datetime64[us]
 8   day_of_week               20000 non-null  str           
 9   is_sunday                 20000 non-null  bool          
 10  proxy_attendance_flag     20000 non-null  bool          
 11  school_id                 20000 non-null  str           
 12  teacher_present           18831 no

In [60]:
print("Final shape:", attendance_final.shape)
print("Duplicate rows:", attendance_final.duplicated().sum())
print(
    "Duplicate record IDs:",
    attendance_final["record_id"].duplicated().sum()
)

Final shape: (20000, 13)
Duplicate rows: 0
Duplicate record IDs: 404


In [61]:
print(
    "Duplicate non-missing record IDs:",
    attendance_final.loc[
        attendance_final["record_id"].notna(),
        "record_id"
    ].duplicated().sum()
)

Duplicate non-missing record IDs: 0


In [62]:
print(" FINAL ATTENDANCE VALIDATION ")

print("Raw rows:", len(attendance))
print("Cleaned rows:", len(attendance_final))
print("Rows removed:", len(attendance) - len(attendance_final))

print("Duplicate rows:", attendance_final.duplicated().sum())

print(
    "Duplicate non-missing record IDs:",
    attendance_final.loc[
        attendance_final["record_id"].notna(),
        "record_id"
    ].duplicated().sum()
)

print("Missing school IDs:", attendance_final["school_id"].isna().sum())
print("Invalid dates:", attendance_final["date"].isna().sum())

print(
    "Attendance count anomalies:",
    attendance_final["attendance_count_anomaly"].sum()
)

print(
    "Proxy attendance flags:",
    attendance_final["proxy_attendance_flag"].sum()
)

print("\nMissing values:")
print(attendance_final.isna().sum())

 FINAL ATTENDANCE VALIDATION 
Raw rows: 20800
Cleaned rows: 20000
Rows removed: 800
Duplicate rows: 0
Duplicate non-missing record IDs: 0
Missing school IDs: 0
Invalid dates: 0
Attendance count anomalies: 806
Proxy attendance flags: 979

Missing values:
record_id                    405
grade                          0
total_students                 0
present_students               0
marked_by                   3369
attendance_count_anomaly       0
attendance_rate                0
date                           0
day_of_week                    0
is_sunday                      0
proxy_attendance_flag          0
school_id                      0
teacher_present             1169
dtype: int64


In [63]:
attendance_final.to_csv(
    "../data/processed/attendance_clean.csv",
    index=False
)

print("Cleaned attendance dataset saved successfully!")
print("File: data/processed/attendance_clean.csv")

Cleaned attendance dataset saved successfully!
File: data/processed/attendance_clean.csv
